# v2 ML-решение кейса ВТБ

Ноутбук обучает модель на parquet-датасете из папки `v2` и сохраняет все результаты в `v2/output/`.

## 1. Инициализация

Здесь подключается локальный pipeline из `v2/scripts/analyze_case_v2_ml.py`. Корневые `scripts/` и `output/` не используются.

In [1]:
from pathlib import Path
import json
import sys

cwd = Path.cwd().resolve()
if (cwd / 'data.parquet').exists():
    V2_ROOT = cwd
elif (cwd / 'v2' / 'data.parquet').exists():
    V2_ROOT = cwd / 'v2'
elif (cwd.parent / 'data.parquet').exists():
    V2_ROOT = cwd.parent
else:
    raise RuntimeError('Не нашел папку v2 с data.parquet')

sys.path.insert(0, str(V2_ROOT / 'scripts'))
import analyze_case_v2_ml as pipeline

print('V2_ROOT:', V2_ROOT)
print('Output:', pipeline.OUT)

V2_ROOT: C:\Users\ivglu\Desktop\projects\CASE\v2
Output: C:\Users\ivglu\Desktop\projects\CASE\v2\output


## 2. Загрузка parquet и проверка target

ML ставится на уровне H3: `cell_y = 1`, если зона есть в `target.parquet`, иначе `0`.

In [2]:
pipeline.ensure_dirs()
data, target = pipeline.load_data()

data_h3 = set(data['h3_index'].unique())
target_h3 = set(target['h3_index'].unique())
diagnostic_preview = {
    'data_rows': len(data),
    'target_rows_unique': len(target),
    'data_h3': len(data_h3),
    'positive_h3_in_data': len(data_h3 & target_h3),
    'negative_h3_in_data': len(data_h3 - target_h3),
}
diagnostic_preview

{'data_rows': 4151096,
 'target_rows_unique': 157806,
 'data_h3': 8154,
 'positive_h3_in_data': 1654,
 'negative_h3_in_data': 6500}

## 3. OSM-слой и признаки

OSM кэшируется в `v2/output/osm/`. Если нужен новый запрос к Overpass, в CLI есть флаг `--refresh-osm`.

In [3]:
temp_cell = (
    data[['h3_index']]
    .drop_duplicates()
    .assign(
        lat=lambda df: df['h3_index'].map(lambda idx: pipeline.h3.cell_to_latlng(idx)[0]),
        lon=lambda df: df['h3_index'].map(lambda idx: pipeline.h3.cell_to_latlng(idx)[1]),
    )
)

raw_osm, osm_meta = pipeline.fetch_osm(temp_cell, refresh=False, no_osm=False)
poi = pipeline.osm_to_poi(raw_osm, set(temp_cell['h3_index']))
cell, diagnostics = pipeline.build_cell_features(data, target, poi)

print('POI:', poi.shape)
print('Feature table:', cell.shape)
print('Positive / negative:', diagnostics['h3_positive_cells_in_data'], diagnostics['h3_negative_cells_in_data'])

POI: (8149, 7)
Feature table: (8154, 73)
Positive / negative: 1654 6500


## 4. Обучение ML-модели

`atm_active_customers`, `atm_penetration_observed` и `atm_affinity_mcc_score` не используются как признаки модели, чтобы не допустить target leakage.

In [4]:
model, scored_ml, model_metrics, importances = pipeline.train_model(cell)
scored = pipeline.add_business_scores(scored_ml)

quality = {
    'roc_auc': round(model_metrics['roc_auc'], 4),
    'average_precision': round(model_metrics['average_precision'], 4),
    'precision_top20pct': round(model_metrics['precision_top20pct'], 4),
    'recall_top20pct': round(model_metrics['recall_top20pct'], 4),
    'features_count': len(model_metrics['features']),
    'target_leakage_features_used': sorted(set(model_metrics['features']) & {'atm_active_customers', 'atm_penetration_observed', 'atm_affinity_mcc_score'}),
}
quality

{'roc_auc': 0.8872,
 'average_precision': 0.7255,
 'precision_top20pct': 0.6618,
 'recall_top20pct': 0.6522,
 'features_count': 64,
 'target_leakage_features_used': []}

## 5. Ранжирование зон

Итоговый индекс:

`placement_score = 100 * (0.55 * ml_atm_probability + 0.25 * demand_score_v2 + 0.20 * coverage_gap_score)`

In [5]:
top_cols = ['priority_rank', 'h3_index', 'placement_score', 'ml_atm_probability', 'cell_y', 'recommendation_type', 'why_recommended']
scored[scored['new_atm_candidate']].head(10)[top_cols]

,priority_rank,h3_index,placement_score,ml_atm_probability,cell_y,recommendation_type,why_recommended
5201,1,8911aa7119bffff,93.478213,0.989859,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
4295,2,8911aa6ac3bffff,92.889452,0.982314,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
427,3,891181b6507ffff,92.644389,0.978962,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
4942,4,8911aa70a3bffff,92.461191,0.992268,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
649,5,891181b6cb7ffff,92.283143,0.982040,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
4921,6,8911aa7099bffff,92.207501,0.960638,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
3413,7,8911aa6257bffff,92.202324,0.983499,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
7206,8,8911aa79c63ffff,92.089117,0.970268,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
2206,9,8911aa4cc2fffff,92.017272,0.969014,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...
7212,10,8911aa79c7bffff,91.862471,0.986359,1,Конкурентный кластер без ВТБ,высокая ML-вероятность ATM-сигнала; широкая кл...


## 6. Экспорт артефактов

Сохраняются CSV, графики, HTML-карта, презентация и summary. Все пути находятся внутри `v2/output/`.

In [6]:
artifacts = pipeline.export_artifacts(scored, poi, diagnostics, model_metrics, importances, osm_meta)
charts = pipeline.save_charts(scored, importances, model_metrics)
map_path = pipeline.save_map(scored, poi)
pptx_path = pipeline.save_presentation(scored, diagnostics, model_metrics, charts, artifacts, map_path)
summary_path = pipeline.save_summary(scored, diagnostics, model_metrics, artifacts, map_path, pptx_path)

{
    'summary': summary_path.relative_to(V2_ROOT).as_posix(),
    'map': map_path.relative_to(V2_ROOT).as_posix(),
    'pptx': pptx_path.relative_to(V2_ROOT).as_posix(),
    'shortlist': artifacts['shortlist'].relative_to(V2_ROOT).as_posix(),
    'diagnostics': artifacts['diagnostics'].relative_to(V2_ROOT).as_posix(),
}

{'summary': 'output/v2_solution_summary.md',
 'map': 'output/maps/v2_atm_ml_potential_map.html',
 'pptx': 'output/pptx/vtb_atm_geoanalytics_solution_v2_ml.pptx',
 'shortlist': 'output/data/v2_shortlist_new_atm.csv',
 'diagnostics': 'output/data/v2_diagnostics.json'}

## 7. Контрольная проверка

In [7]:
payload = json.loads(artifacts['diagnostics'].read_text(encoding='utf-8'))
assert payload['case_data']['h3_cells_data'] == len(scored)
assert not quality['target_leakage_features_used']
assert map_path.exists()
assert pptx_path.exists()
assert artifacts['shortlist'].exists()
print('OK: v2 notebook trained ML and saved isolated artifacts.')

OK: v2 notebook trained ML and saved isolated artifacts.
